In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
plt.rcParams['font.family'] = 'Microsoft YaHei'
plt.rcParams['axes.unicode_minus'] = False

In [2]:
df = pd.read_excel(r"C:\Users\adayc\Desktop\code\Drug\DA_data\filtered_DA_daily_meal.xlsx")  
df['DAY'] = df['DAY'].str.extract(r'(\d{1,2}\.\d{1,2})')[0]
df['DAY'] = '2019.' + df['DAY'] 
df['DAY'] = pd.to_datetime(df['DAY'], format='%Y.%m.%d')
base_columns = ['Sampleid', 'DAY', 'TIME']
feature_columns = [col for col in df.columns if col not in base_columns]
sub_tables = {}
for col in feature_columns:
    sub_df = df[base_columns + [col]].copy()
    sub_tables[col] = sub_df
df.columns

Index(['Sampleid', 'DAY', 'TIME', 'c_能量千卡', 'c_蛋白质克', 'c_脂肪克', 'c_碳水化物克',
       'c_膳食纤维克', 'c_胆固醇毫克', 'c_维生素A视黄醇当量μg', 'c_维生素B1毫克', 'c_维生素B2毫克',
       'c_烟酸毫克', 'c_维生素C毫克', 'c_维生素E毫克', 'c_钙毫克', 'c_磷毫克', 'c_钾毫克', 'c_钠毫克',
       'c_镁毫克', 'c_铁毫克', 'c_锌毫克', 'c_硒ug', 'c_铜毫克', 'c_锰毫克'],
      dtype='object')

In [3]:
sub_tables['c_能量千卡'].head()

,Sampleid,DAY,TIME,c_能量千卡
0,DA001,2019-10-29,1,371.288208
1,DA001,2019-10-29,2,1137.871948
2,DA001,2019-10-29,3,863.578125
3,DA001,2019-10-29,4,242.382553
4,DA001,2019-10-30,1,390.833191


In [4]:
meal_df = pd.read_csv(r'C:\Users\adayc\Desktop\code\Drug\reports\results\tables\estimated_meal_times.csv')  
meal_df

,PatientID,Date,Meal1,Meal2,Meal3,Meal4
0,DA001,2019-10-28,08:00,12:05,16:58,19:51
1,DA001,2019-10-29,7:13,11:35,17:00,19:16
2,DA001,2019-10-30,7:20,12:43,17:00,20:02
3,DA001,2019-10-31,08:00,12:00,16:13,22:06
4,DA001,2019-11-01,8:50,12:50,17:36,19:06
...,...,...,...,...,...,...
2362,DA200,2019-11-15,08:00,11:33,16:41,19:33
2363,DA200,2019-11-16,08:00,12:00,17:00,20:22
2364,DA200,2019-11-17,8:03,12:00,17:00,22:11
2365,DA200,2019-11-18,08:00,12:00,17:00,19:03


In [5]:
cgm_folder = r'C:\Users\adayc\Desktop\code\Drug\DA_data\CGM_id_only'
all_cgm_data = []

for filename in os.listdir(cgm_folder):
    if filename.endswith('.txt'):
        file_path = os.path.join(cgm_folder, filename)

        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
        sampleid = lines[0].strip().replace('#', '').strip()  

        cgm_df = pd.read_csv(file_path, sep='\t', skiprows=1, encoding='utf-8')
        cgm_df = cgm_df.rename(columns={'时间': 'Datetime', '葡萄糖历史记录（mmol/L）': 'Glucose'})
        cgm_df['Datetime'] = pd.to_datetime(cgm_df['Datetime'], format='%Y/%m/%d %H:%M')
        cgm_df['Sampleid'] = sampleid
        cgm_df['Date'] = cgm_df['Datetime'].dt.date
        cgm_df['Time'] = cgm_df['Datetime'].dt.strftime('%H:%M')
        all_cgm_data.append(cgm_df[['Sampleid', 'Date', 'Time', 'Glucose']])

# concatenate all CGM data into a single DataFrame
cgm_df = pd.concat(all_cgm_data, ignore_index=True)
print(cgm_df.head())

  Sampleid        Date   Time  Glucose
0    DA001  2019-10-28  10:21      4.3
1    DA001  2019-10-28  10:36      4.5
2    DA001  2019-10-28  10:51      5.1
3    DA001  2019-10-28  11:06      5.0
4    DA001  2019-10-28  11:21      4.4


In [6]:
meal_df['Date'] = pd.to_datetime(meal_df['Date']).dt.date
for nutrient, sub_df in sub_tables.items():
    sub_df['DAY'] = pd.to_datetime(sub_df['DAY']).dt.date

    # combine meal data with meal times
    merged = pd.merge(
        sub_df,
        meal_df,
        left_on=['Sampleid', 'DAY'],
        right_on=['PatientID', 'Date'],
        how='left'
    )

    # sustitute TIME with Meal1, Meal2, Meal3, Meal4
    time_mapping = {
        1: 'Meal1',
        2: 'Meal2',
        3: 'Meal3',
        4: 'Meal4'
    }
    merged['TIME'] = pd.to_numeric(merged['TIME'], errors='coerce')
    for i in range(1, 5):
        merged.loc[merged['TIME'] == i, 'TIME'] = merged.loc[merged['TIME'] == i, f'Meal{i}']

    keep_cols = ['Sampleid', 'DAY', 'TIME', nutrient]
    sub_tables[nutrient] = merged[keep_cols].copy()

C:\Users\adayc\AppData\Local\Temp\ipykernel_14916\445719360.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['7:13' '7:20' '08:00' ... nan nan nan]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  merged.loc[merged['TIME'] == i, 'TIME'] = merged.loc[merged['TIME'] == i, f'Meal{i}']
C:\Users\adayc\AppData\Local\Temp\ipykernel_14916\445719360.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['7:13' '7:20' '08:00' ... nan nan nan]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  merged.loc[merged['TIME'] == i, 'TIME'] = merged.loc[merged['TIME'] == i, f'Meal{i}']
C:\Users\adayc\AppData\Local\Temp\ipykernel_14916\445719360.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future versio

In [8]:
# Delete no meal data
dfc = sub_tables['c_能量千卡']
zero_indices = dfc.index[np.isclose(dfc['c_能量千卡'], 0, atol=1e-6)].tolist()
for key in sub_tables:
    sub_tables[key] = sub_tables[key].drop(index=zero_indices)

In [11]:
sub_tables['c_能量千卡']

,Sampleid,DAY,TIME,c_能量千卡
0,DA001,2019-10-29,7:13,371.288208
1,DA001,2019-10-29,11:35,1137.871948
2,DA001,2019-10-29,17:00,863.578125
3,DA001,2019-10-29,19:16,242.382553
4,DA001,2019-10-30,7:20,390.833191
...,...,...,...,...
8978,DA200,2019-11-28,NaN,344.850891
8979,DA200,2019-11-28,NaN,457.431000
8980,DA200,2019-11-29,NaN,439.875763
8981,DA200,2019-11-29,NaN,237.449997


In [12]:
output_dir = r'C:\Users\adayc\Desktop\code\Drug\reports\results\tables\subtables'
for key in sub_tables:
    filename = sub_tables[key].columns[3]
    filepath = os.path.join(output_dir, f"{filename}.csv")
    sub_tables[key].to_csv(filepath, index=False)

print(f"Subtables saved to {output_dir}")

Subtables saved to C:\Users\adayc\Desktop\code\Drug\reports\results\tables\subtables
